# mine-tuning-model

## 필수 라이브러리 설치

In [13]:
!pip install --upgrade chromadb langchain langchain-community langchain-openai sentence-transformers transformers datasets gradio langgraph -q

ERROR: Operation cancelled by user


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip freeze > /content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model/requirements.txt

## 허깅페이스에서 데이터 가져오기

In [ ]:
from datasets import load_dataset

ds = load_dataset("naklecha/minecraft-question-answer-700k")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['answer', 'question', 'source'],
        num_rows: 694814
    })
})


# 데이터 임베딩


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print('[입베딩 성공]')
print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[입베딩 성공]
임베딩 차원: 384


/tmp/ipykernel_8982/1680344745.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"임베딩 차원: {embedding_model.get_sentence_embedding_dimension()}")


## Chroma DB 구축

In [ ]:
import chromadb
from tqdm import tqdm

COLLECTION_NAME = "minecraft_rag"
CHROMA_DIR      = "/content/chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
existing      = [c.name for c in chroma_client.list_collections()]

# 기존 컬렉션 있고 데이터도 있으면 재사용
if COLLECTION_NAME in existing:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    if collection.count() > 0:
        print(f"✅ 기존 chroma_db 로드 완료: {collection.count()}개 데이터")
    else:
        # 데이터 0개면 삭제 후 새로 구축
        print("⚠️ 데이터 0개 — 삭제 후 새로 구축합니다...")
        chroma_client.delete_collection(name=COLLECTION_NAME)
        existing = []  # 아래 else 블록 타도록 초기화

if COLLECTION_NAME not in existing:
    # 🆕 새로 구축
    print("[ChromaDB 새로 구축 중...]")
    collection = chroma_client.create_collection(name=COLLECTION_NAME)

    total_size = len(ds["train"]["answer"])
    answers    = ds["train"]["answer"]
    questions  = ds["train"]["question"]
    sources    = ds["train"]["source"]
    print(f"총 데이터 수: {total_size}개")

    # ============================================================
    # [테스트용 5000개 - 주석 처리]
    # sample_size = 5000
    # batch_size  = 500
    # for i in range(0, sample_size, batch_size):
    #     batch_answers = answers[i:i+batch_size]
    #     embeddings = embedding_model.encode(batch_answers, show_progress_bar=True).tolist()
    #     collection.add(
    #         documents=batch_answers,
    #         embeddings=embeddings,
    #         metadatas=[{"question": q, "source": s} for q, s in zip(questions[i:i+batch_size], sources[i:i+batch_size])],
    #         ids=[str(j) for j in range(i, i+len(batch_answers))]
    #     )
    #     print(f" {min(i+batch_size, sample_size)}/{sample_size} 완료")
    # ============================================================

    # 전체 데이터 구축
    batch_size = 500
    for i in tqdm(range(0, total_size, batch_size), desc="벡터 DB 구축 중"):
        batch_answers   = answers[i:i+batch_size]
        batch_questions = questions[i:i+batch_size]
        batch_sources   = sources[i:i+batch_size]

        embeddings = embedding_model.encode(
            batch_answers,
            show_progress_bar=False
        ).tolist()

        collection.add(
            documents=batch_answers,
            embeddings=embeddings,
            metadatas=[{"question": q, "source": s} for q, s in zip(batch_questions, batch_sources)],
            ids=[str(j) for j in range(i, i+len(batch_answers))]
        )

    print(f"\n🎉 완료! 총 {collection.count()}개 저장됨")

⚠️ 데이터 0개 — 삭제 후 새로 구축합니다...
[ChromaDB 새로 구축 중...]
총 데이터 수: 694814개


벡터 DB 구축 중: 100%|██████████| 1390/1390 [29:44<00:00,  1.28s/it]



🎉 완료! 총 694814개 저장됨


## 검색 테스트

In [16]:
def retrieve(query: str, top_k: int = 3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

# 테스트
docs, metas = retrieve("How to find diamonds?")
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"=== 검색 결과 {i+1} ===")
    print(f"source     : {meta['source']}")
    print(f"Q       : {meta['question']}")
    print(f"Answer  : {doc[:200]}")
    print()

=== 검색 결과 1 ===
source     : https://minecraft.wiki/w/Character_Creator
Q       : What is the best way to get a diamond in Minecraft?
Answer  : A diamond can be obtained by mining deep into the game's caves and ravines. It is also possible to find diamonds by exploring abandoned mineshafts and pillaging the chests found within.

=== 검색 결과 2 ===
source     : https://minecraft.wiki/w/Tutorials/Advancement_guide/Minecraft_tab
Q       : What is a reliable method to obtain diamonds in Minecraft?
Answer  : One reliable method to obtain diamonds is to mine deep down in the world, specifically between Y=-58, or explore caves that go very low. You can also look for diamonds in various loot chests, such as 

=== 검색 결과 3 ===
source     : https://minecraft.wiki/w/1.16-pre7
Q       : How do I get a diamond in Minecraft?
Answer  :  You can find diamonds by mining deep into caves or ravines. Look for exposed diamond ore and mine it with a pickaxe. You can also find diamonds in abandoned mineshafts or

## LLM 로드

In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False
)

print("LLM 로드 완료")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM 로드 완료


## RAG 파이프라인 완성

In [18]:
def rag_answer(query: str) -> str:
    # 1. 관련 문서 검색
    docs, metas = retrieve(query, top_k=3)
    context = "\n\n".join(docs)

    # 2. 프롬프트 구성
    prompt = f"""You are a helpful Minecraft expert assistant.
Use the following context to answer the question accurately.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""

    # 3. LLM 답변 생성
    messages = [{"role": "user", "content": prompt}]
    result = llm(messages)
    answer = result[0]["generated_text"][-1]["content"]

    # 4. 출처 출력
    print(f"출처: {metas[0]['source']}\n")
    return answer


In [19]:
# 테스트
response = rag_answer("How to make a Nether portal?")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


출처: https://minecraft.wiki/w/Boss_mob

To make a Nether portal, you need to craft a Nether portal first and then right-click on the ground. Alternatively, you can create it by building a Nether portal with obsidian blocks and flint and steel, placing them in the correct positions as described in the context.


## Gradio 연결

In [20]:
import gradio as gr

def chat(message, history):
    answer = rag_answer(message)
    return answer

demo = gr.ChatInterface(
    fn=chat,
    title="⛏️ Minecraft Guide LLM",
    description="Minecraft Wiki 기반 AI 가이드에게 무엇이든 물어보세요!",
    examples=[
        "How to find diamonds?",
        "How do I defeat the Ender Dragon?",
        "How to make a Nether portal?"
    ],
)

demo.launch(share=True)  # share=True 로 외부 접속 링크 생성

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d6b5d522372785c0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
